In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

dbutils.widgets.removeAll()
dbutils.widgets.text("storage_name", "adlsproject1225")
dbutils.widgets.text("container", "data")
dbutils.widgets.text("catalog", "adventure_works")
dbutils.widgets.text("schema_bronze", "bronze")
dbutils.widgets.text("schema_silver", "silver")
dbutils.widgets.text("schema_golden", "golden")


SECRET_SCOPE = "accessScopeforADLS"
KEY_CLIENT_ID = "databricks-app-client-id2"
KEY_TENANT_ID = "databricks-app-tenant-id2"
KEY_CLIENT_SECRET = "databricks-app-client-secret2"


In [0]:
storage_name = dbutils.widgets.get("storage_name")
container = dbutils.widgets.get("container")
catalog = dbutils.widgets.get("catalog")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_silver = dbutils.widgets.get("schema_silver")
schema_golden = dbutils.widgets.get("schema_golden")

client_id = dbutils.secrets.get(SECRET_SCOPE, KEY_CLIENT_ID)
tenant_id = dbutils.secrets.get(SECRET_SCOPE, KEY_TENANT_ID)
client_secret = dbutils.secrets.get(SECRET_SCOPE, KEY_CLIENT_SECRET)

print("Using storage:", storage_name, "container:", container, "catalog:", catalog)


In [0]:
spark.conf.set(f"fs.azure.account.auth.type.{storage_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_name}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")

spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_name}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_name}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")


In [0]:
spark.sql(f"""
CREATE OR REPLACE STORAGE CREDENTIAL cred_adls
WITH AZURE_SERVICE_PRINCIPAL (
  CLIENT_ID='{client_id}',
  CLIENT_SECRET='{client_secret}',
  TENANT_ID='{tenant_id}'
)
COMMENT 'Credential created from Environment Preparation notebook';
""")
print("Storage credential created/replaced: cred_adls")


In [0]:
spark.sql(f"""
CREATE EXTERNAL LOCATION IF NOT EXISTS exlt_raw
URL 'abfss://raw@{storage_name}.dfs.core.windows.net/'
WITH (STORAGE_CREDENTIAL cred_adls)
COMMENT 'Raw files (CSV)';
""")

spark.sql(f"""
CREATE EXTERNAL LOCATION IF NOT EXISTS exlt_bronze
URL 'abfss://bronze@{storage_name}.dfs.core.windows.net/'
WITH (STORAGE_CREDENTIAL cred_adls)
COMMENT 'Bronze Delta tables location';
""")

spark.sql(f"""
CREATE EXTERNAL LOCATION IF NOT EXISTS exlt_silver
URL 'abfss://silver@{storage_name}.dfs.core.windows.net/'
WITH (STORAGE_CREDENTIAL cred_adls)
COMMENT 'Silver Delta tables location';
""")

spark.sql(f"""
CREATE EXTERNAL LOCATION IF NOT EXISTS exlt_golden
URL 'abfss://golden@{storage_name}.dfs.core.windows.net/'
WITH (STORAGE_CREDENTIAL cred_adls)
COMMENT 'Golden Delta tables location';
""")


spark.sql(f"""
CREATE CATALOG IF NOT EXISTS {catalog}
MANAGED LOCATION 'abfss://{container}@{storage_name}.dfs.core.windows.net/';
""")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_bronze};")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_silver};")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_golden};")
